# Lecture 2a — Typing: annotating the code you write

This lecture adds one important notion to your programming skills: **type checking**.
It is an important tool for software reliability: it lets you verify that every function
is called correctly and that it operates on the correct datatypes.

This is provided in the Python language, but the principle applies to other languages as
well (with some technical differences).

> **Lecture 2 comes in two halves.** This one, **2a**, is about *writing* annotations:
> the vocabulary of the type system and how to put it on the functions and data you
> already have. **[Lecture 2b](Lecture2b.ipynb)** is about *reasoning* with
> the type system — subtypes and variance, generics with `TypeVar`, classes as types —
> and closes with a tour of the `typing` module.
>
> Read 2a, do **Labwork 2, exercises 1 and 2**; then read 2b and do exercises 3 and 4.
> Exercise 3 (a generic linked list) is unreachable without 2b's `TypeVar`.


## Objectives
By the end of this half you can:
- explain **dynamic vs static vs duck** typing, and why type hints have **no runtime effect**;
- annotate variables, parameters and return values, including `None` and `NoReturn`;
- type composite data (`list`, `tuple`, `dict`) and define **type aliases**;
- use `Any`, and say why annotating with `Any` is close to not annotating at all;
- run **`mypy`** on a file and read what it prints.

Subtypes, variance, `TypeVar`, `Optional`, classes as types and the `typing` module tour
are all in **Lecture 2b**.


## How to read this lecture

| Section | Read | What it gives you |
|---|---|---|
| 1 · Type systems | 5 min | the three words — dynamic, static, duck |
| 2 · Hello types | 5 min | your first annotated function, and `mypy` finding a real bug |
| 3 · A deck of cards | 11 min | sequences, mappings, aliases, `None`, `Any` |
| **End to end** | **≈ 25 min** | enough for Labwork 2, exercises 1–2 |

(Reading time only — add a few minutes for running the cells.)

Run the code cells as you go — the two `%%python` cells run the card game in a
subprocess, and the rest run in the notebook itself. Reading about typing without
watching `mypy` complain is the one way to come out of this lecture with nothing.


## Internet links
Typing was introduced into Python version 3.5 thanks to the [PEP 483](https://peps.python.org/pep-0483/) and [PEP 484](https://peps.python.org/pep-0484/). 
Notice that PEP means *Python Enhancement Proposals*. 
The full history of the different proposals (done, accepted, under consideration or rejected) is available [here](https://peps.python.org/topic/typing/).

While it is possible to simply read the PEP 484 at first, we choose in this lecture a more didactical approach following [this link](https://realpython.com/python-type-checking/)...

## Type Systems
All programming languages include some kind of type system that formalizes which categories of objects it can work with and how those categories are treated. For instance, a type system can define a numerical type, with `42` as one example of an object of numerical type.

### Dynamic typing
**Python is a dynamically typed language**: the Python interpreter does type checking only as code runs, and that the type of a variable is allowed to change over its lifetime. The following dummy examples demonstrate that Python has dynamic typing:
```python
>>> if False:
...     1 + "two"  # This line never runs, so no TypeError is raised
... else:
...     1 + 2
...
3

>>> 1 + "two"  # Now this is type checked, and a TypeError is raised
TypeError: unsupported operand type(s) for +: 'int' and 'str'
```
In the first example, the branch `1 + "two"` never runs so it’s never type checked. The second example shows that when `1 + "two"` is evaluated it raises a `TypeError` since you can’t add an integer and a string in Python.

Next, let’s see if variables can change type:
```python
>>> thing = "Hello"
>>> type(thing)
<class 'str'>

>>> thing = 28.1
>>> type(thing)
<class 'float'>
```
In Python, `type()` returns the type of an object. These examples confirm that the type of thing is allowed to change, and Python correctly infers the type as it changes.

### Static typing
The opposite of dynamic typing is **static typing**. Static type checks are performed without running the program, generally as your program is compiled.

With static typing, variables generally are not allowed to change types, although mechanisms for casting a variable to a different type may exist.

Let’s look at a quick example from a statically typed language. Consider the following Java snippet:
```Java
String thing;
thing = "Hello";
```
The first line declares that the variable name thing is bound to the `String` type at compile time. The name can never be rebound to another type. In the second line, thing is assigned a value. It can never be assigned a value that is not a `String` object. For instance, if you were to later say `thing = 28.1f` the compiler would raise an error because of incompatible types.

Python will always remain a dynamically typed language. However, PEP 484 introduced type hints, which make it possible to also do static type checking of Python code.

Unlike how types work in most other statically typed languages, type hints by themselves don’t cause Python to enforce types. As the name says, type hints just suggest types. There are other tools, which you’ll see later, that perform static type checking using type hints.

### Duck typing
Another term that is often used when talking about Python is duck typing. This moniker comes from the phrase “if it walks like a duck and it quacks like a duck, then it must be a duck” (or any of its variations).

Duck typing is a concept related to dynamic typing, where the type or the class of an object is less important than the methods it defines. Using duck typing you do not check types at all. Instead you check for the presence of a given method or attribute.

As an example, you can call `len()` on any Python object that defines a `.__len__()` method:
```python
>>> class TheHobbit:
...     def __len__(self):
...         return 95022
...
>>> the_hobbit = TheHobbit()
>>> len(the_hobbit)
95022
```
Note that the call to `len()` gives the return value of the `.__len__()` method. In fact, the implementation of `len()` is essentially equivalent to the following:
```python
def len(obj):
    return obj.__len__()
```
In order to call `len(obj)`, the only real constraint on obj is that it must define a `.__len__()` method. Otherwise, the object can be of types as different as `str`, `list`, `dict`, or `TheHobbit`.

Duck typing is somewhat supported when doing static type checking of Python code, using structural subtyping. You’ll learn more about duck typing later.

In [ ]:
# Dynamic typing, live. Nothing here is checked until the line actually runs.
if False:
    1 + "two"          # never runs, so never type checked -- no error
else:
    print(1 + 2)

thing = "Hello"
print(type(thing))     # <class 'str'>
thing = 28.1
print(type(thing))     # <class 'float'> -- the SAME name, a different type

try:
    1 + "two"          # now it runs, and only now does Python object
except TypeError as err:
    print("TypeError:", err)


## Hello Types
This section presents how to add type hints to a function. 
The following function displays a hello message using the name given in parameters:
```python
def greeting(name, capitalized = True):
    if capitalized:
        return 'Hello ' + name
    return 'hello ' + name
```

It’s time for our first type hints! To add information about types to the function, you simply annotate its arguments and return value as follows:
```python
def greeting(name: str, capitalized: bool = True) -> str:
    ...
```
The text: `str` syntax says that the argument should be of type `str`. Similarly, the optional `capitalized` argument should have type bool with the default value `True`. Finally, the `-> str` notation specifies that the function returns a string.

In terms of style, PEP 8 recommends the following:
- Use normal rules for colons, that is, no space before and one space after a colon: `name: str`.
- Use spaces around the `=` sign when combining an argument annotation with a default value: `capitalized: bool = True`.
- Use spaces around the `->` arrow: `def headline(...) -> str`.

Adding type hints like this has no runtime effect: they are only hints and are not enforced on their own. For instance, if we use a wrong type for the (admittedly badly named) `capitalized` argument, the code still runs without any problems or warnings:
```python
print(greeting("Marc Antoine", capitalized = "yes"))
```
displays
```python
Hello Marc Antoine
```
The reason this seemingly works is that the string `"yes"` compares as truthy. Using `capitalized = "no"` would not have the desired effect as `"no"` is also truthy:
```python
print(greeting("Marc Antoine", capitalized = "no"))
```
displays
```python
Hello Marc Antoine
```

To catch this kind of error you can use a static type checker. That is, a tool that checks the types of your code without actually running it in the traditional sense.

You might already have such a type checker built into your editor. For instance PyCharm immediately gives you a warning.
The most common tool for doing type checking is `mypy` though. You’ll get a short introduction to `mypy` in a moment, while you can learn much more about how it works later.

If you don’t already have `mypy` on your system, you can install it using `pip`:
```bash
$ python -m pip install mypy
```
Put the following code in a file called greeting.py:
```python
def greeting(name: str, capitalized: bool = True) -> str:
    if capitalized:
        return 'Hello ' + name
    return 'hello ' + name

print(greeting('Marc Antoine', capitalized = "yes"))
```
This is essentially the same code you saw earlier: the definition of `greeting()` and one examples that is using it.

Now run `mypy` on this code:
```bash
$ mypy headlines.py
greeting.py:6: error: Argument "capitalized" to "greeting" has incompatible type "str"; expected "bool"
Found 1 error in 1 file (checked 1 source file)
```
Based on the type hints, `mypy` is able to tell us that we are using the wrong type on line 6.

To fix the issue in the code you should change the value of the align argument you are passing in. By doing so, the error raised by `mypy` disappears...

In [ ]:
def greeting(name: str, capitalized: bool = True) -> str:
    if capitalized:
        return "Hello " + name
    return "hello " + name


# `capitalized` is annotated `bool`, but Python does not care at run time:
print(greeting("Marc Antoine", capitalized="yes"))   # "yes" is truthy  -> "Hello ..."
print(greeting("Marc Antoine", capitalized="no"))    # "no"  is ALSO truthy -> "Hello ..."

# The annotations are just data hanging off the function object:
print(greeting.__annotations__)

# `name` is annotated `str`. Passing an int is a type error that mypy reports without
# running anything -- but Python lets the call through, and only blows up later, inside
# the function, on the `+`. That gap between "where mypy points" and "where it crashes"
# is the whole argument for static checking.
try:
    print(greeting(42))
except TypeError as err:
    print("TypeError:", err)


### Run `mypy` yourself

Everything above was a transcript. Now produce it. The three cells below write the
`greeting.py` from the section above into a scratch folder and check it — the same
`%%writefile` + `!mypy --strict` rhythm the labworks use.

`mypy` should report **exactly one error**, on the line that passes `capitalized="yes"`.
Note what that means: the program above *ran perfectly*. No exception, no wrong answer
that day. The checker is telling you about a bug that has not happened yet.

> Needs `mypy` installed — `python -m pip install mypy` if the cell says
> `command not found`.


In [ ]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable
os.makedirs(".dummy", exist_ok=True)
print(".dummy/ ready")
print(f'interpreter: {PY}')


In [ ]:
%%writefile .dummy/greeting.py
def greeting(name: str, capitalized: bool = True) -> str:
    if capitalized:
        return "Hello " + name
    return "hello " + name


print(greeting("Marc Antoine", capitalized="yes"))


In [ ]:
!{PY} .dummy/greeting.py
!{PY} -m mypy --strict .dummy/greeting.py


Two outputs, and the contrast between them is the whole point of this lecture:

```
Hello Marc Antoine                                       <- python: happy
.dummy/greeting.py:7: error: Argument "capitalized" to "greeting" has incompatible
    type "str"; expected "bool"  [arg-type]              <- mypy: not happy
Found 1 error in 1 file (checked 1 source file)
```

Fix the call to `capitalized=True` (or `False`) and re-run the two cells: `mypy` prints
`Success: no issues found in 1 source file`. From Unit 2 onwards, **that** is the bar
every labwork must clear.


## Playing with typing - 1
Up until now you’ve only saw how to use basic types like `str`, `float`, or `bool` in your type hints. The Python type system is quite powerful, and supports many kinds of more complex types. This is necessary as it needs to be able to reasonably model Python’s dynamic duck typing nature.

This section introduces more about this type system, while implementing a simple card game. You will see how to specify:
- The type of sequences and mappings like tuples, lists and dictionaries.
- Type aliases that make code easier to read.
- That functions and methods do not return anything.
- Objects that may be of any type.

### Example: A Deck of Cards
The following example shows an implementation of a regular (French) deck of cards:

In [ ]:
%%script {PY}
# game.py
import random

SUITS = "♠ ♡ ♢ ♣".split()
RANKS = "2 3 4 5 6 7 8 9 10 J Q K A".split()

def create_deck(shuffle=False):
    """Create a new deck of 52 cards"""
    deck = [(s, r) for r in RANKS for s in SUITS]
    if shuffle:
        random.shuffle(deck)
    return deck

def deal_hands(deck):
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])

def play():
    """Play a 4-player card game"""
    deck = create_deck(shuffle=True)
    names = "P1 P2 P3 P4".split()
    hands = {n: h for n, h in zip(names, deal_hands(deck))}

    for name, cards in hands.items():
        card_str = " ".join(f"{s}{r}" for (s, r) in cards)
        print(f"{name}: {card_str}")

if __name__ == "__main__":
    play()

Each card is represented as a tuple of strings denoting the suit and rank. The deck is represented as a list of cards. `create_deck()` creates a regular deck of 52 playing cards, and optionally shuffles the cards. `deal_hands()` deals the deck of cards to four players.

Finally, `play()` plays the game. As of now, it only prepares for a card game by constructing a shuffled deck and dealing cards to each player. 

You will see how to extend this example into a more interesting game as we move along.

### Sequences and Mappings
Let’s add type hints to our card game. In other words, let’s annotate the functions `create_deck()`, `deal_hands()`, and `play()`. The first challenge is that you need to annotate composite types like the list used to represent the deck of cards and the tuples used to represent the cards themselves.

With simple types like `str`, `float`, and `bool`, adding type hints is as easy as using the type itself:
```python
name: str = "Guido"
pi: float = 3.142
centered: bool = False
```
With composite types, you are allowed to do the same:
```python
names: list = ["Guido", "Jukka", "Ivan"]
version: tuple = (3, 7, 1)
options: dict = {"centered": False, "capitalize": True}
```
However, this does not really tell the full story. What will be the types of `names[2]`, `version[0]`, and `options["centered"]`? In this concrete case you can see that they are `str`, `int`, and `bool`, respectively. However, the type hints themselves give no information about this.

Instead, you should use the special types defined in the typing module. These types add syntax for specifying the types of elements of composite types. You can write the following:
```python
from typing import Dict, List, Tuple

names: List[str] = ["Guido", "Jukka", "Ivan"]
version: Tuple[int, int, int] = (3, 7, 1)
options: Dict[str, bool] = {"centered": False, "capitalize": True}
```
Note that each of these types start with a capital letter and that they all use square brackets to define item types:
- `names` is a list of strings.
- `version` is a 3-tuple consisting of three integers.
- `options` is a dictionary mapping strings to `Boolean` values.

The typing module contains many more composite types, including `Counter`, `Deque`, `FrozenSet`, `NamedTuple`, and `Set`. In addition, the module includes other kinds of types that you’ll see in later sections.

Let’s return to the card game. A card is represented by a tuple of two strings. You can write this as `Tuple[str, str]`, so the type of the deck of cards becomes `List[Tuple[str, str]]`. Therefore you can annotate `create_deck()` as follows:
```python
def create_deck(shuffle: bool = False) -> List[Tuple[str, str]]:
    """Create a new deck of 52 cards"""
    deck = [(s, r) for r in RANKS for s in SUITS]
    if shuffle:
        random.shuffle(deck)
    return deck
```
In addition to the returned value, you’ve also added the `bool` type to the optional shuffle argument.

In many cases your functions will expect some kind of sequence, and not really care whether it is a list or a tuple. In these cases you should use `typing.Sequence` when annotating the function argument:
```python
from typing import List, Sequence

def square(elems: Sequence[float]) -> List[float]:
    return [x**2 for x in elems]
```
Using `Sequence` is an example of using duck typing. A `Sequence` is anything that supports `len()` and `.__getitem__()`, independent of its actual type. Quack!

> ### Note — `List[str]` or `list[str]`?
>
> The capitalised `List`, `Dict`, `Tuple`, `Set` from `typing` are what you will see in
> most tutorials, including the card game below, and they are what the rest of this
> lecture uses. Since **Python 3.9** the builtin types are generic themselves, and since
> **3.10** unions have their own syntax, so the modern spelling is shorter:
>
> | Old (`typing`) | Modern (builtin) |
> |---|---|
> | `List[str]` | `list[str]` |
> | `Dict[str, bool]` | `dict[str, bool]` |
> | `Tuple[int, int, int]` | `tuple[int, int, int]` |
> | `Optional[str]` | `str \| None` |
> | `Union[int, str]` | `int \| str` |
>
> This course requires **Python ≥ 3.11**, so you may write either — `mypy --strict`
> accepts both. Prefer the modern form in code you write; recognise the old form,
> because you will read it constantly. Only `Sequence`, `Mapping`, `Callable`,
> `Iterator` and friends still need an import (from `collections.abc`, not `typing`).


### Type Aliases
The type hints might become quite oblique when working with nested types like the deck of cards. You may need to stare at `List[Tuple[str, str]]` a bit before figuring out that it matches our representation of a deck of cards.

Now consider how you would annotate `deal_hands()`:
```python
def deal_hands(
    deck: List[Tuple[str, str]]
) -> Tuple[
    List[Tuple[str, str]],
    List[Tuple[str, str]],
    List[Tuple[str, str]],
    List[Tuple[str, str]],
]:
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])
```
That’s just terrible!

Recall that type annotations are regular Python expressions. That means that you can define your own type aliases by assigning them to new variables. You can for instance create `Card`, `Deck` and `Hand` type aliases:
```python
from typing import List, Tuple

Card = Tuple[str, str]
Deck = List[Card]
Round = Tuple[Deck, Deck, Deck, Deck]
```
`Card` can now be used in type hints or in the definition of new type aliases, like `Deck` and `Hand` in the example above.

Using these aliases, the annotations of `deal_hands()` become much more readable:
```python
def deal_hands(deck: Deck) -> Round:
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])
```
Type aliases are great for making your code and its intent clearer. At the same time, these aliases can be inspected to see what they represent:
```python
from typing import List, Tuple
Card = Tuple[str, str]
Deck = List[Card]
Deck
```
that will display:
```python
typing.List[typing.Tuple[str, str]]
```
Note that when printing `Deck`, it shows that it’s an alias for a list of 2-tuples of strings.

### Functions Without Return Values
You may know that functions without an explicit return still return `None`:
```python
def play(player_name):
     print(f"{player_name} plays")
ret_val = play("Jacob")
print(ret_val)
```
that will display:
```python
Jacob plays
None
```
While such functions technically return something, that return value is not useful. You should add type hints saying as much by using `None` also as the return type:
```python
# play.py
def play(player_name: str) -> None:
    print(f"{player_name} plays")
ret_val = play("Filip")
```
The annotations help catch the kinds of subtle bugs where you are trying to use a meaningless return value. `Mypy` will give you a helpful warning:
```bash
$ mypy play.py
play.py:4: error: "play" does not return a value
```
Note that being explicit about a function not returning anything is different from not adding a type hint about the return value:
```python
# play.py
def play(player_name: str):
    print(f"{player_name} plays")
ret_val = play("Henrik")
```
In this latter case `mypy` has no information about the return value so it will not generate any warning:
```bash
$ mypy play.py
Success: no issues found in 1 source file
```
As a more exotic case, note that you can also annotate functions that are never expected to return normally. This is done using `NoReturn`:
```python
from typing import NoReturn
def black_hole() -> NoReturn:
    raise Exception("There is no going back ...")
```
Since `black_hole()` always raises an exception, it will never return properly.



### Example: Play Some Cards
Let’s return to our card game example. In this second version of the game, we deal a hand of cards to each player as before. Then a start player is chosen and the players take turns playing their cards. There are not really any rules in the game though, so the players will just play random cards:

In [ ]:
%%script {PY}
import random
from typing import List, Tuple

SUITS = "♠ ♡ ♢ ♣".split()
RANKS = "2 3 4 5 6 7 8 9 10 J Q K A".split()

Card = Tuple[str, str]
Deck = List[Card]

def create_deck(shuffle: bool = False) -> Deck:
    """Create a new deck of 52 cards"""
    deck = [(s, r) for r in RANKS for s in SUITS]
    if shuffle:
        random.shuffle(deck)
    return deck

def deal_hands(deck: Deck) -> Tuple[Deck, Deck, Deck, Deck]:
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])

def choose(items):
    """Choose and return a random item"""
    return random.choice(items)

def player_order(names, start=None):
    """Rotate player order so that start goes first"""
    if start is None:
        start = choose(names)
    start_idx = names.index(start)
    return names[start_idx:] + names[:start_idx]

def play() -> None:
    """Play a 4-player card game"""
    deck = create_deck(shuffle=True)
    names = "P1 P2 P3 P4".split()
    hands = {n: h for n, h in zip(names, deal_hands(deck))}
    start_player = choose(names)
    turn_order = player_order(names, start=start_player)

    # Randomly play cards from each player's hand until empty
    while hands[start_player]:
        for name in turn_order:
            card = choose(hands[name])
            hands[name].remove(card)
            print(f"{name}: {card[0] + card[1]:<3}  ", end="")
        print()

if __name__ == "__main__":
    play()

### The Any Type
Note that in addition to changing play(), we have added two new functions that need type hints: `choose()` and `player_order()`. 

`choose()` works for both lists of names and lists of cards (and any other sequence for that matter). One way to add type hints for this would be the following:
```python
import random
from typing import Any, Sequence

def choose(items: Sequence[Any]) -> Any:
    return random.choice(items)
```
This means more or less what it says: items is a sequence that can contain items of any type and `choose()` will return one such item of any type. Unfortunately, this is not that useful. Consider the following example:
```python
# choose.py
import random
from typing import Any, Sequence

def choose(items: Sequence[Any]) -> Any:
    return random.choice(items)

names = ["Guido", "Jukka", "Ivan"]
reveal_type(names)

name = choose(names)
reveal_type(name)
```
The `reveal_type()` primitive is injected by type checkers into the builtins. 
When the type checker sees a call, it prints the inferred type of the argument (as a `note:` line). 

While `mypy` correctly infers that names is a list of strings, that information is lost after the call to `choose()` because of the use of the `Any` type:
```bash
$ mypy choose.py
choose.py:10: note: Revealed type is "builtins.list[builtins.str]"
choose.py:13: note: Revealed type is "Any"
```
You will see a better way shortly. First though, let’s have a more theoretical look at the Python type system, and the special role `Any` plays.

## Checkpoint

You can now annotate a function: its parameters, its return value, `None` when it
returns nothing, and the shape of the lists, tuples and dictionaries it passes around.
You can hide a complicated shape behind a type alias, and you can run `mypy` and read
what it says. You also know that `Any` switches the checker off for whatever it touches.

That is exactly what **Labwork 2, exercises 1 and 2** ask for — annotating `power`, and
re-annotating the `Vegetable` class from Unit 1. Do them now, before reading on.

Two questions are still open, and both matter:

1. `choose()` above works on a list of names *or* a list of cards, and annotating it with
   `Any` threw away everything the checker knew. What is the honest annotation?
2. `player_order(names, start=None)` takes a `str` **or** `None`. Neither annotation
   alone is right.

**[Lecture 2b](Lecture2b.ipynb)** answers both — the first with `TypeVar`, the second
with `Optional` — after a short look at what "subtype" actually means.
